# 10 · Behavioral transfer — Personalizing an AV with a human driver's style

The decomposition is *generative*: once a trajectory is split into a common
context and a personalized style, the style of one agent can be grafted onto the
context of another.

**Part A — Personalization from data.** `data/synthetic/behavioral_transfer_data.xlsx`
holds a leader speed profile, a target HDV following it, and a baseline AV
following it. We transfer the HDV's style (its residual) onto the AV's common
context to obtain a *personalized AV* that keeps the AV's context but drives with
the human's style.

**Part B — Multiple driver archetypes with safety controls.** Four archetypes
(timid, aggressive, smooth "limo", and a *defensive* variant obtained by shifting
the aggressive style earlier in time) are transferred onto the same baseline AV,
with a `safety_gain` and `safety_shift` that let a designer scale or advance the
transferred style.

**Figures produced:** `personalization.png`, `multiple_personalization.png`.

In [ ]:
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "bhd").exists())
sys.path.insert(0, str(ROOT / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.ndimage import shift
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

from bhd import DATA_DIR, FIG_DIR, set_style

set_style()

## Part A — Style transfer from the speed profiles in the data file

In [ ]:
def get_physics_properties(v, t, dt, start_pos=0):
    a = np.gradient(v, dt)
    x = start_pos + np.cumsum(v) * dt
    return x, v, a


def transfer_style(v_source, v_target):
    """Graft the *source* (HDV) style onto the *target* (AV) common context."""
    X = np.column_stack([v_source, v_target])
    scaler = StandardScaler()
    X_std = scaler.fit_transform(X)

    pca = PCA(n_components=1)
    C = pca.fit_transform(X_std)
    X_common = pca.inverse_transform(C)
    residuals = X_std - X_common  # style

    av_common = X_common[:, 1]
    hdv_style = residuals[:, 0]
    personalized_std = av_common + hdv_style
    return personalized_std * scaler.scale_[1] + scaler.mean_[1]


df = pd.read_excel(DATA_DIR / "synthetic" / "behavioral_transfer_data.xlsx")
t = df["Time"].values
v_leader = df["Leader_Speed"].values
v_hdv = df["HDV_Speed"].values
v_av = df["AV_Speed"].values
dt = t[1] - t[0]

v_av_personalized = transfer_style(v_hdv, v_av)

x_leader, _, a_leader = get_physics_properties(v_leader, t, dt, start_pos=200)
x_hdv, _, a_hdv = get_physics_properties(v_hdv, t, dt, start_pos=100)
x_av, _, a_av = get_physics_properties(v_av, t, dt, start_pos=100)
x_av_personalized, _, a_av_personalized = get_physics_properties(v_av_personalized, t, dt, start_pos=100)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(20, 7))


def style_ax(ax, title, ylabel):
    ax.set_title(title, fontsize=22)
    ax.set_ylabel(ylabel, fontsize=20)
    ax.set_xlabel("Time (s)", fontsize=20)
    ax.tick_params(axis="both", which="major", labelsize=18)
    ax.grid(True, linestyle="-", alpha=0.3)


for ax, (L, AV, HDV, PER) in zip(
    axes,
    [(x_leader, x_av, x_hdv, x_av_personalized), (v_leader, v_av, v_hdv, v_av_personalized), (a_leader, a_av, a_hdv, a_av_personalized)],
):
    ax.plot(t, L, "k--", linewidth=3, label="Leader", alpha=0.5)
    ax.plot(t, AV, "b:", linewidth=3, label="Baseline AV", alpha=0.6)
    ax.plot(t, HDV, "g--", linewidth=5, label="Target HDV", alpha=0.4)
    ax.plot(t, PER, "m-", linewidth=4, label="Personalized AV")
style_ax(axes[0], "Trajectory", "Position (m)")
style_ax(axes[1], "Speed Profile", "Speed (m/s)")
style_ax(axes[2], "Acceleration Profile", "Accel (m/s²)")
axes[0].legend(fontsize=18, loc="upper left")
plt.tight_layout()
plt.savefig(FIG_DIR / "personalization.png", dpi=300, bbox_inches="tight")
plt.show()

## Part B — Multiple archetypes with safety gain / shift

In [ ]:
def generate_archetype(t, archetype):
    v0 = 20.0
    if archetype == "standard_av":
        return v0 - 12 * np.exp(-0.012 * (t - 30) ** 2)
    elif archetype == "timid":
        return v0 - 8 * np.exp(-0.005 * (t - 22) ** 2)
    elif archetype == "aggressive":  # sharp, deep, late
        return v0 - 16 * np.exp(-0.04 * (t - 35) ** 2)
    elif archetype == "limo":  # wide and smooth
        return v0 - 8 * np.exp(-0.003 * (t - 30) ** 2)
    raise ValueError(archetype)


def get_acceleration(v, t, dt):
    return np.gradient(v, dt)


def transfer_style_safe(v_source, v_target, safety_gain=1.0, safety_shift=0.0, v_equilibrium=20.0):
    """Style transfer in deviation space with a gain and a time shift on the style."""
    delta_source = v_source - v_equilibrium
    delta_target = v_target - v_equilibrium

    X = np.column_stack([delta_source, delta_target])
    scaler = StandardScaler(with_mean=False)
    X_std = scaler.fit_transform(X)

    pca = PCA(n_components=1)
    C = pca.fit_transform(X_std)
    X_common = pca.inverse_transform(C)
    residuals = X_std - X_common

    hdv_style_safe = residuals[:, 0] * safety_gain
    av_context = X_common[:, 1]
    personalized_std = av_context + hdv_style_safe

    delta_personalized = scaler.inverse_transform(np.column_stack([np.zeros_like(personalized_std), personalized_std]))[:, 1]

    if safety_shift != 0:
        delta_personalized -= delta_personalized[-1]  # leveling
        dt = 60 / 400
        delta_personalized = shift(delta_personalized, int(safety_shift / dt), mode="constant", cval=0.0)

    return v_equilibrium + delta_personalized


t = np.linspace(0, 60, 400)
dt = t[1] - t[0]
v0 = 20.0

experiments = [
    {"name": "(a) Timid", "hdv": "timid", "gain": 1.0, "shift": 0.0},
    {"name": "(b) Aggressive", "hdv": "aggressive", "gain": 1.0, "shift": 0.0},
    {"name": "(c) Smooth", "hdv": "limo", "gain": 1.0, "shift": 0.0},
    {"name": "(d) Defensive", "hdv": "aggressive", "gain": 1.0, "shift": -5.0},
]

v_av_baseline = generate_archetype(t, "standard_av")
a_av_baseline = get_acceleration(v_av_baseline, t, dt)

results = []
for exp in experiments:
    v_hdv = generate_archetype(t, exp["hdv"])
    v_personalized = transfer_style_safe(v_hdv, v_av_baseline, safety_gain=exp["gain"], safety_shift=exp["shift"], v_equilibrium=v0)
    results.append({"meta": exp, "a_hdv": get_acceleration(v_hdv, t, dt), "a_personalized": get_acceleration(v_personalized, t, dt)})

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
for i, ax in enumerate(axes.flatten()):
    res = results[i]
    ax.plot(t, a_av_baseline, "b:", linewidth=3, label="Baseline AV", alpha=0.6)
    ax.plot(t, res["a_hdv"], "g--", linewidth=5, label="Target HDV", alpha=0.4)
    ax.plot(t, res["a_personalized"], "m-", linewidth=4, label="Personalized AV")
    ax.set_title(res["meta"]["name"], fontsize=22)
    ax.set_ylabel("Accel ($m/s^2$)", fontsize=20)
    ax.set_xlabel("Time (s)", fontsize=20)
    ax.tick_params(axis="both", which="major", labelsize=18)
    ax.grid(True, linestyle="-", alpha=0.3)
    if i == 0:
        ax.legend(loc="lower right", fontsize=18, frameon=True, framealpha=0.9)
plt.tight_layout()
plt.savefig(FIG_DIR / "multiple_personalization.png", dpi=300, bbox_inches="tight")
plt.show()